In [1]:
import numpy as np
import torch
import os
import sys
import yaml
from sklearn.metrics import roc_auc_score
helpers_path = os.path.join('/ether/aegis/Research_HEP/NRAD/model_scripts')
sys.path.insert(0, os.path.abspath(helpers_path))
from Classifier import Classifier
# from SimpleMAF import SimpleMAF

In [2]:
seed = 2
data_path = f"SemiVisJets/data/data_seed{seed}"
samples_path = "SemiVisJets/samples"
eval_path = "SemiVisJets/eval_sr"

In [3]:
def regularize_weights(w_arr, sigma = 3.0):
    w_copy = np.copy(w_arr)
    mean_w = np.mean(w_copy)
    std_w = np.std(w_copy)
    w_copy[w_copy > (sigma*std_w + mean_w)] = 0
    return w_copy

In [4]:
def run_eval(set_1, set_2, code, save_dir, classifier_params, device, w_1 = None, w_2 = None, crop_weights = True, classifier_runs = 20):
    
    if w_1 is None:
        w_1 = np.array([1.]*set_1.shape[0])
    if w_2 is None:
        w_2 = np.array([1.]*set_2.shape[0])
    if crop_weights:
        w_1 = regularize_weights(w_1)
        w_2 = regularize_weights(w_2)
    
    num_test = min(10000, set_1.shape[0] // 5)

    trainset_1, testset_1 = set_1[:-num_test], set_1[-num_test:]
    trainset_2, testset_2 = set_2[:-num_test], set_2[-num_test:]

    wtrain_1, wtest_1 = w_1[:-num_test], w_1[-num_test:]
    wtrain_2, wtest_2 = w_2[:-num_test], w_2[-num_test:]

    # input_x_train = np.concatenate([set_1, set_2], axis=0)
    # input_y_train = np.concatenate([np.zeros(set_1.shape[0]).reshape(-1,1), np.ones(set_2.shape[0]).reshape(-1,1)], axis=0)
    # input_w_train = np.concatenate([w_1, w_2], axis=0).reshape(-1, 1)

    # ---------- Build train/test sets ----------
    input_x_train = np.concatenate([trainset_1, trainset_2], axis=0)
    input_y_train = np.concatenate([
        np.zeros(trainset_1.shape[0]),
        np.ones(trainset_2.shape[0])
    ], axis=0).reshape(-1, 1)
    
    input_w_train = np.concatenate([wtrain_1, wtrain_2], axis=0).reshape(-1, 1)


    input_x_test = np.concatenate([testset_1, testset_2], axis=0)
    input_y_test = np.concatenate([
        np.zeros(testset_1.shape[0]),
        np.ones(testset_2.shape[0])
    ], axis=0).reshape(-1, 1)
    
    # ---------- Logging ----------
    print(f"\nWorking on {code}...")
    print("      X_train, y_train, w_train:", input_x_train.shape, input_y_train.shape, input_w_train.shape)
    print("      X_test, y_test:", input_x_test.shape, input_y_test.shape)
    

    # if run_test:
    #     input_x_test = np.concatenate([test_B, test_S], axis=0)
    #     input_y_test = np.concatenate([np.zeros(test_B.shape[0]).reshape(-1,1), np.ones(test_S.shape[0]).reshape(-1,1)], axis=0)
    #     print("      X test, y test:", input_x_test.shape, input_y_test.shape)
    aucs_list = []
    for i in range(int(classifier_runs)):
        
        print(f"Classifier run {i+1} of {classifier_runs}.")
        local_id = f"{code}_run{i}"
                
        # train classifier
        NN = Classifier(n_inputs=5, layers=classifier_params["layers"], learning_rate=classifier_params["learning_rate"], device=device, scale_data=False)
        NN.train(input_x_train, input_y_train, weights=input_w_train,  save_model=True, model_name = f"model_{local_id}" , n_epochs=classifier_params["n_epochs"], seed = i, outdir=save_dir)

        # if run_test:
        scores = NN.evaluation(input_x_test)
        auc = roc_auc_score(input_y_test, scores, sample_weight=np.concatenate([wtest_1, wtest_2]))
        if auc < 0.5:
            auc = 1.0 - auc  # symmetry adjustment
        aucs_list.append(auc)
        print(f"   AUC: {auc}")
    
    os.makedirs(f"{save_dir}/auc_scores", exist_ok=True)
    np.savez(f"{save_dir}/auc_scores/auc_{code}.npz", auc_scores=np.array(aucs_list))

    print("\nMedian AUC, 16th percentile, 84th percentile:")
    print(np.median(aucs_list), [np.percentile(aucs_list, 16), np.percentile(aucs_list, 84)])
    print("Done.\n")

In [5]:
print("Setting up device...")
CUDA = torch.cuda.is_available()
print("cuda available:", CUDA)
device = torch.device("cuda" if CUDA else "cpu")

Setting up device...
cuda available: True


In [6]:
# Load in the classifier params
config_path = "oldver/NRAD/non-resonant-AD/configs"
with open(f"{config_path}/bc_discrim.yml", 'r') as stream:
    params = yaml.safe_load(stream)

n_context = 2

In [7]:
print("Training CWoLa for Reweight Samples on SR data")
for i in range(1, 6):
    reweights_events = np.load(f"{samples_path}/reweight_MC{seed:02d}_Data{i:02d}_SR_samples.npz", allow_pickle=True)
    data_events = np.load(f"SemiVisJets/data/data_test/data_events_chunk{6:02d}.npz", allow_pickle=True)
    set_1 = reweights_events['mc_samples'][:, n_context:]
    set_2 = data_events["data_events_sr"][:, n_context:]
    w_1 = reweights_events['w_sr']
    run_eval(set_1, set_2, code = f"reweight_SR_Data{i:02d}_MC    {seed:02d}", save_dir=eval_path, classifier_params=params, device=device, crop_weights=True)
    print(f"Reweight on Data{i:02d} @ MC{seed:02d}") 

Training CWoLa for Reweight Samples on SR data

Working on reweight_SR_Data01_MC    02...
      X_train, y_train, w_train: (44547, 5) (44547, 1) (44547, 1)
      X_test, y_test: (18786, 5) (18786, 1)
Classifier run 1 of 20.


 46%|====>     | 23/50 [00:09<00:11,  2.33it/s]


   AUC: 0.6907852704063743
Classifier run 2 of 20.


 60%|======    | 30/50 [00:12<00:08,  2.35it/s]


   AUC: 0.6932003190943548
Classifier run 3 of 20.


...

   AUC: 0.692635370933473
Classifier run 19 of 20.


 54%|=====     | 27/50 [00:12<00:10,  2.20it/s]


   AUC: 0.6924540348657835
Classifier run 20 of 20.


 46%|====>     | 23/50 [00:10<00:12,  2.17it/s]


   AUC: 0.6909911110417457

Median AUC, 16th percentile, 84th percentile:
0.6918060522632559 [0.6909920404471804, 0.6927918918760546]
Done.

Reweight on Data05 @ MC02


In [8]:
print("Training CWoLa for Generate Samples on SR data")
for i in range(1, 6):
    generate_events = np.load(f"{samples_path}/generate_MC{seed:02d}_Data{i:02d}_SR_samples.npz", allow_pickle=True)
    context_weights = np.load(f"{samples_path}/context_weight_MC{seed:02d}_Data{i:02d}_SR_samples.npz", allow_pickle=True)
    data_events = np.load(f"SemiVisJets/data/data_test/data_events_chunk{6:02d}.npz", allow_pickle=True)
    set_1 = generate_events['samples']
    set_2 = data_events["data_events_sr"][:, n_context:]
    w_1 = context_weights['w_sr']
    run_eval(set_1, set_2, code = f"generate_SR_Data{i:02d}_MC    {seed:02d}", save_dir=eval_path, classifier_params=params, device=device, crop_weights=True)
    print(f"Generate on Data{i:02d} @ MC{seed:02d}") 

Training CWoLa for Generate Samples on SR data

Working on generate_SR_Data01_MC    02...
      X_train, y_train, w_train: (44547, 5) (44547, 1) (44547, 1)
      X_test, y_test: (18786, 5) (18786, 1)
Classifier run 1 of 20.


 46%|====>     | 23/50 [00:10<00:12,  2.12it/s]


   AUC: 0.6744117705163332
Classifier run 2 of 20.


 56%|=====>    | 28/50 [00:12<00:09,  2.31it/s]


   AUC: 0.6756699417894108
Classifier run 3 of 20.


...

   AUC: 0.6933596044513941
Classifier run 19 of 20.


 80%|========  | 40/50 [00:17<00:04,  2.30it/s]


   AUC: 0.6960041029396312
Classifier run 20 of 20.


 74%|=======   | 37/50 [00:14<00:05,  2.51it/s]


   AUC: 0.6941755374165084

Median AUC, 16th percentile, 84th percentile:
0.6941646849079258 [0.6915073783060608, 0.6964778102355625]
Done.

Generate on Data05 @ MC02
